In [10]:
from fractions import Fraction

def eval_poly(poly, x):
    """
    Avalia o polinômio P(x) = a0 + a1 x + a2 x^2 + ...
    onde poly = [a0, a1, a2, ...]
    """
    x = Fraction(x)
    value = Fraction(0)
    pot = Fraction(1)  # x^0
    for coef in poly:
        value += coef * pot
        pot *= x
    return value


def verificar_polinomio(xs, ys, poly, usar_float=False, tol=1e-9):
    """
    Verifica se P(xi) = yi para todos os pontos.
    - xs: lista de x_i
    - ys: lista de f(x_i)
    - poly: lista de coeficientes (Fractions ou floats)
    - usar_float: se True, compara com tolerância (útil se coeficientes forem float)
    - tol: tolerância para comparação em ponto flutuante
    """
    print("Verificando se o polinômio interpola os pontos:")
    tudo_ok = True

    for xi, yi in zip(xs, ys):
        valor = eval_poly(poly, xi)

        if usar_float:
            # comparação com tolerância (caso esteja usando floats)
            dif = float(valor) - float(yi)
            ok = abs(dif) <= tol
        else:
            # comparação exata (Fraction)
            ok = (valor == yi)

        status = "OK ✅" if ok else "ERRO ❌"
        print(f"x = {xi:>4} : P(x) = {valor} ; f(x) = {yi}  ->  {status}")

        if not ok:
            tudo_ok = False

    if tudo_ok:
        print("\nResultado: o polinômio está CORRETO para todos os pontos. 🎉")
    else:
        print("\nResultado: o polinômio NÃO interpola todos os pontos. ⚠️")

    return tudo_ok

# Operações básicas com polinômios (lista de coeficientes)
# p = [a0, a1, a2, ...]  =>  a0 + a1 x + a2 x^2 + ...

def poly_add(p, q):
    m = max(len(p), len(q))
    r = [Fraction(0)] * m
    for i in range(m):
        if i < len(p):
            r[i] += p[i]
        if i < len(q):
            r[i] += q[i]
    return r

def poly_mul(p, q):
    r = [Fraction(0)] * (len(p) + len(q) - 1)
    for i, ai in enumerate(p):
        for j, bj in enumerate(q):
            r[i + j] += ai * bj
    return r

def poly_to_str(poly, var='x'):
    """Transforma [a0, a1, a2] em string tipo '1 - 7/3x + 2/3x^2'."""
    terms = []
    for i, coef in reversed(list(enumerate(poly))):
        if coef == 0:
            continue
        sign = "-" if coef < 0 else "+"
        abs_coef = -coef if coef < 0 else coef

        if abs_coef == 1 and i != 0:
            coef_str = ""
        else:
            coef_str = str(abs_coef)

        if i == 0:
            term = coef_str
        elif i == 1:
            term = f"{coef_str}{var}" if coef_str else var
        else:
            term = f"{coef_str}{var}^{i}" if coef_str else f"{var}^{i}"

        terms.append((sign, term))

    if not terms:
        return "0"

    first_sign, first_term = terms[0]
    s = ("-" if first_sign == "-" else "") + first_term
    for sign, term in terms[1:]:
        s += f" {sign} {term}"
    return s


def newton_step_by_step(xs, ys, var='x'):
    n = len(xs)

    # ===== 1) Tabela de diferenças divididas =====
    table = [[Fraction(0) for _ in range(n)] for _ in range(n)]
    for i in range(n):
        table[i][0] = Fraction(ys[i])

    print("Cálculo das diferenças divididas:")
    for ordem in range(1, n):
        print(f"\nOrdem {ordem}:")
        for i in range(n - ordem):
            num = table[i + 1][ordem - 1] - table[i][ordem - 1]
            den = Fraction(xs[i + ordem] - xs[i])
            table[i][ordem] = num / den
            print(
                f"f[x{i},...,x{i+ordem}] = "
                f"({table[i + 1][ordem - 1]} - {table[i][ordem - 1]}) "
                f"/ ({xs[i + ordem]} - {xs[i]}) = {table[i][ordem]}"
            )

    # Imprime tabela no estilo do slide
    print("\nTabela de diferenças divididas:")
    header = ["x"] + [f"ordem {k}" for k in range(n)]
    print(" | ".join(f"{h:^10}" for h in header))
    print("-" * (13 * len(header)))
    for i in range(n):
        row = [f"{xs[i]:^10}"]
        for k in range(n - i):
            row.append(f"{str(table[i][k]):^10}")
        print(" | ".join(row))

    # ===== 2) Forma de Newton simbólica =====
    print("\nForma geral de Newton (com os valores calculados):")
    partes = []
    for k in range(n):
        coef = table[0][k]
        if k == 0:
            partes.append(f"f[x0] = {coef}")
        else:
            fatores = "".join([f"({var} - {xs[j]})" for j in range(k)])
            partes.append(f"{fatores} * f[x0,...,x{k}] = {fatores} * ({coef})")
    print("P(x) = " + " + ".join(partes))

    # ===== 3) Construção P0, P1, P2,... passo a passo =====
    print("\nConstrução passo a passo do polinômio:")
    poly = [Fraction(0)]

    for k in range(n):
        coef = table[0][k]

        # base_k(x) = (x - x0)(x - x1)...(x - x_{k-1})
        if k == 0:
            base = [Fraction(1)]
            termo_poly = [coef]
            termo_str = str(coef)
        else:
            base = [Fraction(1)]
            for j in range(k):
                # (x - x_j)   =>  [-x_j, 1]
                base = poly_mul(base, [Fraction(-xs[j]), Fraction(1)])
            termo_poly = [coef * c for c in base]
            fatores = " * ".join([f"({var} - {xs[j]})" for j in range(k)])
            termo_str = f"{fatores} * ({coef})"

        print(f"\nTermo {k}:")
        if k == 0:
            print(f"T_{k}(x) = f[x0] = {termo_str}")
        else:
            print(f"Base_{k}(x) = " + " * ".join([f"({var} - {xs[j]})" for j in range(k)]))
            print(f"T_{k}(x) = Base_{k}(x) * f[x0,...,x{k}] = {termo_str}")

        # soma P_{k-1}(x) + T_k(x)
        poly = poly_add(poly, termo_poly)
        print(f"P_{k}(x) = {poly_to_str(poly, var)}")

    print("\nPolinômio final expandido:")
    print("P(x) = " + poly_to_str(poly, var))

    return table, poly


    xs = [-2, 0, 3, 4, 5]
    ys = [-10, 2, 20, 50, 102]

tabela, coeficientes = newton_step_by_step(xs, ys)


Cálculo das diferenças divididas:

Ordem 1:
f[x0,...,x1] = (2 - -10) / (0 - -2) = 6
f[x1,...,x2] = (20 - 2) / (3 - 0) = 6
f[x2,...,x3] = (50 - 20) / (4 - 3) = 30
f[x3,...,x4] = (102 - 50) / (5 - 4) = 52

Ordem 2:
f[x0,...,x2] = (6 - 6) / (3 - -2) = 0
f[x1,...,x3] = (30 - 6) / (4 - 0) = 6
f[x2,...,x4] = (52 - 30) / (5 - 3) = 11

Ordem 3:
f[x0,...,x3] = (6 - 0) / (4 - -2) = 1
f[x1,...,x4] = (11 - 6) / (5 - 0) = 1

Ordem 4:
f[x0,...,x4] = (1 - 1) / (5 - -2) = 0

Tabela de diferenças divididas:
    x      |  ordem 0   |  ordem 1   |  ordem 2   |  ordem 3   |  ordem 4  
------------------------------------------------------------------------------
    -2     |    -10     |     6      |     0      |     1      |     0     
    0      |     2      |     6      |     6      |     1     
    3      |     20     |     30     |     11    
    4      |     50     |     52    
    5      |    102    

Forma geral de Newton (com os valores calculados):
P(x) = f[x0] = -10 + (x - -2) * f[x0,...,x1] = 

In [2]:
# -*- coding: utf-8 -*-
from sympy import symbols, Rational, expand, latex, fraction
from IPython.display import display, Math


def _tabela_diferencas_divididas(xs, ys):
    """
    Constrói a tabela de diferenças divididas de Newton.
    Retorna uma lista de linhas: dd[k][i] = f[x_i,...,x_{i+k}]
    """
    n = len(xs)
    dd = [[None]*(n-k) for k in range(n)]

    # primeira linha: f[x_i] = y_i
    for i in range(n):
        dd[0][i] = ys[i]

    # demais linhas
    for k in range(1, n):
        for i in range(n-k):
            dd[k][i] = (dd[k-1][i+1] - dd[k-1][i]) / (xs[i+k] - xs[i])

    return dd


def newton_divididas_latex(xs, ys, var_name="x"):
    """
    Interpolação de Newton (diferenças divididas) com saída em LaTeX,
    no estilo "resolvendo à mão".

    xs: lista de x_i (todos distintos)
    ys: lista de f(x_i)
    var_name: nome da variável simbólica, ex: "x"
    """
    # variável simbólica
    x = symbols(var_name)

    # converte para frações exatas
    xs = [Rational(v) for v in xs]
    ys = [Rational(v) for v in ys]

    n = len(xs)
    if n != len(ys):
        raise ValueError("xs e ys devem ter o mesmo tamanho.")
    if len(set(xs)) != n:
        raise ValueError("Os x_i devem ser todos distintos.")

    # tabela de diferenças divididas
    dd = _tabela_diferencas_divididas(xs, ys)

    print("Pontos de interpolação:")
    for i, (xi, yi) in enumerate(zip(xs, ys)):
        print(f"  (x_{i}, f(x_{i})) = ({xi}, {yi})")
    print()

    # mostra tabela numérica (triangular) em texto
    print("Tabela de diferenças divididas (f[x_i, ..., x_{i+k}]):")
    for k in range(n):
        linha = ", ".join(str(dd[k][i]) for i in range(n-k))
        print(f"  ordem {k}: {linha}")
    print()

    # coeficientes a_k = f[x_0, ..., x_k] = primeira entrada de cada linha
    a = [dd[k][0] for k in range(n)]

    # forma geral do polinômio de Newton
    display(Math(
        r"P_{%d}(%s) = a_0 + a_1(%s - x_0)"
        r" + a_2(%s - x_0)(%s - x_1) + \cdots"
        % (n-1, var_name, var_name, var_name, var_name)
    ))

    # mostra a_k em LaTeX
    for k in range(n):
        display(Math(
            r"a_%d = f[x_0,\ldots,x_%d] = %s"
            % (k, k, latex(a[k]))
        ))

    # construção dos termos e do polinômio
    P = 0
    termos = []

    for k in range(n):
        # produto (x - x_0)(x - x_1)...(x - x_{k-1})
        prod = 1
        prod_tex = ""
        for j in range(k):
            fator = (x - xs[j])
            prod *= fator
            prod_tex += r"(%s - %s)" % (var_name, latex(xs[j]))

        Tk = a[k] * prod  # termo de ordem k

        display(Math(r"\textbf{Termo }k=%d" % k))

        if k == 0:
            # T_0(x) = a_0
            display(Math(
                r"T_0(%s) = a_0 = %s"
                % (var_name, latex(a[k]))
            ))
        else:
            # mostra produto e depois o termo
            display(Math(
                r"\prod_{j=0}^{%d-1}(%s - x_j) = %s = %s"
                % (k, var_name, prod_tex, latex(expand(prod)))
            ))
            display(Math(
                r"T_{%d}(%s) = a_%d \cdot %s = %s \cdot %s"
                % (k, var_name, k, prod_tex or "1",
                   latex(a[k]), latex(expand(prod)))
            ))

            num_Tk, den_Tk = fraction(Tk)
            num_Tk_exp = expand(num_Tk)

            display(Math(
                r"T_{%d}(%s) = \frac{%s}{%s}"
                % (k, var_name, latex(num_Tk_exp), latex(den_Tk))
            ))
            display(Math(
                r"T_{%d}(%s) = %s"
                % (k, var_name, latex(expand(Tk)))
            ))

        P += Tk
        termos.append(Tk)

    # polinômio final
    P_exp = expand(P)
    num_P, den_P = fraction(P_exp)
    num_P_exp = expand(num_P)

    display(Math(r"\textbf{Polinômio final de Newton (diferenças divididas)}"))
    display(Math(r"P(%s) = %s" % (var_name, latex(P_exp))))
    if den_P != 1:
        display(Math(
            r"P(%s) = \dfrac{%s}{%s}"
            % (var_name, latex(num_P_exp), latex(den_P))
        ))

    return P_exp


# ====================== EXEMPLO DE USO ======================
if __name__ == "__main__":
    # EXECUÇÂO
    xs = [-2, 0, 3, 4, 5]
    ys = [-10, 2, 20, 50, 102]

    P = newton_divididas_latex(xs, ys)
    print("P(x) =", P)


Pontos de interpolação:
  (x_0, f(x_0)) = (-2, -10)
  (x_1, f(x_1)) = (0, 2)
  (x_2, f(x_2)) = (3, 20)
  (x_3, f(x_3)) = (4, 50)
  (x_4, f(x_4)) = (5, 102)

Tabela de diferenças divididas (f[x_i, ..., x_{i+k}]):
  ordem 0: -10, 2, 20, 50, 102
  ordem 1: 6, 6, 30, 52
  ordem 2: 0, 6, 11
  ordem 3: 1, 1
  ordem 4: 0



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

P(x) = x**3 - x**2 + 2
